# HRV Stress Detection — Data Mining & ML Coursework (F21DL)

**Dataset:** HRV stress dataset — RRI beat-level recordings + minute-level labeled sessions
(conditions: rest, no stress, time pressure, interruption) across 25 participants.

**Pipeline:**
1. Data cleaning & merging
2. Feature engineering
3. Exploratory Data Analysis
4. Generative modeling (Naive Bayes, GMM/clustering)
5. Discriminative modeling (Decision Tree, Logistic Regression, etc.)
6. Evaluation & comparison


## 1. Data Cleaning & Merging

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("hrv dataset/data/raw")
LABELS_FILE = RAW_DIR / "labels" / "hrv stress labels.xlsx"
RRI_DIR = RAW_DIR / "rri"
PROCESSED_DIR = Path("hrv dataset/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Labels: one sheet per participant (p1..p25), stacked into one table
labels_sheets = pd.read_excel(LABELS_FILE, sheet_name=None)
labels_df = pd.concat(labels_sheets.values(), ignore_index=True)
print("Raw labels shape:", labels_df.shape)
labels_df.head()

Raw labels shape: (3140, 12)


,Unnamed: 0,PP,C,timestamp,HR,RMSSD,SCL,date,subject,label,Condition,ElapsedTime
0,0,PP1,1,20120918T131600000,NaN,NaN,80.239727,2012-09-18 13:16:00,p1,rest,R,0
1,1,PP1,1,20120918T131700000,61.0,0.061420,77.365127,2012-09-18 13:17:00,p1,rest,R,1
2,2,PP1,1,20120918T131800000,64.0,0.049663,77.359559,2012-09-18 13:18:00,p1,rest,R,2
3,3,PP1,1,20120918T131900000,60.0,0.052487,76.728772,2012-09-18 13:19:00,p1,rest,R,3
4,4,PP1,1,20120918T132000000,61.0,0.051189,76.512877,2012-09-18 13:20:00,p1,rest,R,4


### Known data quality issues
- `timestamp` string column is malformed for 7 rows (truncated, e.g. `20121001T162`) — the parsed
  `date` column (datetime64) is reliable for all rows, so we drop `timestamp` and use `date`.
- `HR` and `RMSSD` are missing for 1,650/3,140 rows (~52%). This is **not random**: subjects
  `p8` and `p11` have **100%** missing HR/RMSSD because they have no corresponding RRI file
  (HR/RMSSD are derived from beat-to-beat RR intervals — no ECG recording, no derived values).
- `SCL` (skin conductance level) varies by orders of magnitude between subjects (e.g. p1 ~80 vs
  p9 ~1400) — this is a per-subject baseline/sensor-placement effect, not signal; we normalise
  per-subject with a z-score.


In [2]:
clean = labels_df.drop(columns=["Unnamed: 0", "timestamp"]).copy()
clean = clean.rename(columns={"PP": "pp_code", "C": "session"})

print("Missing values per column:")
print(clean.isna().sum())
clean.to_csv(PROCESSED_DIR / "labels_clean.csv", index=False)

Missing values per column:
pp_code           0
session           0
HR             1650
RMSSD          1650
SCL             562
date              0
subject           0
label             0
Condition         0
ElapsedTime       0
dtype: int64


In [3]:
# Impute HR/RMSSD/SCL: per-subject median first, global median fallback
# (fallback only matters for p8/p11, which have zero RRI-derived signal)
global_med = {c: clean[c].median() for c in ["HR", "RMSSD", "SCL"]}

for col in ["HR", "RMSSD", "SCL"]:
    subj_med = clean.groupby("subject")[col].transform("median")
    clean[f"{col}_imputed_flag"] = clean[col].isna().astype(int)
    clean[f"{col}_filled"] = clean[col].fillna(subj_med).fillna(global_med[col])

# Per-subject z-score for SCL (fixes wildly different baselines across subjects)
clean["SCL_z"] = clean.groupby("subject")["SCL_filled"].transform(lambda s: (s - s.mean()) / s.std())

final_cols = ["subject", "pp_code", "session", "date", "ElapsedTime", "Condition", "label",
              "HR_filled", "RMSSD_filled", "SCL_filled", "SCL_z",
              "HR_imputed_flag", "RMSSD_imputed_flag", "SCL_imputed_flag"]
features_minute = clean[final_cols].rename(
    columns={"HR_filled": "HR", "RMSSD_filled": "RMSSD", "SCL_filled": "SCL"}
)

print("Final shape:", features_minute.shape)
print(features_minute["label"].value_counts())
features_minute.to_csv(PROCESSED_DIR / "features_minute_level.csv", index=False)
features_minute.head()

Final shape: (3140, 14)
label
no stress        1029
interruption      996
time pressure     664
rest              451
Name: count, dtype: int64


,subject,pp_code,session,date,ElapsedTime,Condition,label,HR,RMSSD,SCL,SCL_z,HR_imputed_flag,RMSSD_imputed_flag,SCL_imputed_flag
0,p1,PP1,1,2012-09-18 13:16:00,0,R,rest,65.0,0.048305,80.239727,3.337706,1,1,0
1,p1,PP1,1,2012-09-18 13:17:00,1,R,rest,61.0,0.061420,77.365127,2.680570,0,0,0
2,p1,PP1,1,2012-09-18 13:18:00,2,R,rest,64.0,0.049663,77.359559,2.679298,0,0,0
3,p1,PP1,1,2012-09-18 13:19:00,3,R,rest,60.0,0.052487,76.728772,2.535099,0,0,0
4,p1,PP1,1,2012-09-18 13:20:00,4,R,rest,61.0,0.051189,76.512877,2.485746,0,0,0


### RRI (beat-level) files
25 participants expected; `p8` and `p11` have no RRI file (confirmed above — matches 100% missing
HR/RMSSD for those subjects). Raw RRI time (`time_s`) is seconds since each participant's
recording start and does **not** line up cleanly with the labels' `ElapsedTime` once you account
for breaks between sessions (recording pauses during breaks, elapsed-time clock does not) — so we
use the RRI files for supplementary signal visualisation (tachogram, Poincare plot) rather than
row-level merging into the modeling table.

In [4]:
rri_files = sorted(RRI_DIR.glob("p*.txt"), key=lambda p: int(p.stem[1:]))
print(f"{len(rri_files)} RRI files found:", [f.stem for f in rri_files])

rri_frames = []
for path in rri_files:
    frame = pd.read_csv(path, sep=r"\s+", header=None, names=["time_s", "rri_ms"])
    frame.insert(0, "subject", path.stem)
    rri_frames.append(frame)
rri_df = pd.concat(rri_frames, ignore_index=True)
print("RRI combined shape:", rri_df.shape)
rri_df.to_csv(PROCESSED_DIR / "rri_combined.csv", index=False)
rri_df.head()

23 RRI files found: ['p1', 'p2', 'p3', 'p4', 'p5', 'p6', 'p7', 'p9', 'p10', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25']
RRI combined shape: (704454, 3)


,subject,time_s,rri_ms
0,p1,1.265625,870.11719
1,p1,1.515625,885.36996
2,p1,1.765625,890.18974
3,p1,2.015625,886.73851
4,p1,2.265625,877.17820


## 2. Feature Engineering

### Target definitions
- **Multi-class target**: `label` (4 classes: rest, no stress, time pressure, interruption)
- **Binary target**: `is_stress` — stressors (time pressure, interruption) = 1, baseline
  (rest, no stress) = 0. Based on the SWELL-style experiment design where time pressure and
  interruption are the stress-inducing conditions.

### Derived features (minute-level table)
- `SCL_log` = log1p(SCL) — SCL is heavy-right-tailed (max 1455 vs median 172), log-transform
  stabilises variance for linear/generative models.
- `RMSSD_HR_ratio` — rough parasympathetic/sympathetic balance proxy.
- `label_code` — integer encoding of the 4-class label for models needing numeric targets.


In [5]:
features_minute = pd.read_csv(PROCESSED_DIR / "features_minute_level.csv")

stress_labels = {"time pressure", "interruption"}
features_minute["is_stress"] = features_minute["label"].isin(stress_labels).astype(int)
features_minute["SCL_log"] = np.log1p(features_minute["SCL"])
features_minute["RMSSD_HR_ratio"] = features_minute["RMSSD"] / features_minute["HR"]
features_minute["label_code"] = features_minute["label"].astype("category").cat.codes

features_minute.to_csv(PROCESSED_DIR / "features_engineered.csv", index=False)
print("Engineered shape:", features_minute.shape)
print("\nBinary class balance (is_stress):")
print(features_minute["is_stress"].value_counts())
print("\nMulti-class balance (label):")
print(features_minute["label"].value_counts())
features_minute.head()

Engineered shape: (3140, 18)

Binary class balance (is_stress):
is_stress
1    1660
0    1480
Name: count, dtype: int64

Multi-class balance (label):
label
no stress        1029
interruption      996
time pressure     664
rest              451
Name: count, dtype: int64


,subject,pp_code,session,date,ElapsedTime,Condition,label,HR,RMSSD,SCL,SCL_z,HR_imputed_flag,RMSSD_imputed_flag,SCL_imputed_flag,is_stress,SCL_log,RMSSD_HR_ratio,label_code
0,p1,PP1,1,2012-09-18 13:16:00,0,R,rest,65.0,0.048305,80.239727,3.337706,1,1,0,0,4.397404,0.000743,2
1,p1,PP1,1,2012-09-18 13:17:00,1,R,rest,61.0,0.061420,77.365127,2.680570,0,0,0,0,4.361379,0.001007,2
2,p1,PP1,1,2012-09-18 13:18:00,2,R,rest,64.0,0.049663,77.359559,2.679298,0,0,0,0,4.361308,0.000776,2
3,p1,PP1,1,2012-09-18 13:19:00,3,R,rest,60.0,0.052487,76.728772,2.535099,0,0,0,0,4.353225,0.000875,2
4,p1,PP1,1,2012-09-18 13:20:00,4,R,rest,61.0,0.051189,76.512877,2.485746,0,0,0,0,4.350444,0.000839,2


### RRI-derived subject HRV profiles (supplementary)
Per the Step-1 alignment limitation, we don't merge beat-level RRI into the per-minute table.
Instead we compute one aggregate HRV profile per subject (whole recording) — useful for EDA,
tachogram/Poincare plots, and an extra unsupervised angle (e.g. clustering subjects by HRV
profile) that doesn't depend on fragile time alignment.

In [6]:
def pnn50(x):
    diffs = np.abs(np.diff(x))
    return 100.0 * np.mean(diffs > 50) if len(diffs) else np.nan

rows = []
for subj, g in rri_df.groupby("subject"):
    vals = g["rri_ms"].values
    rows.append({
        "subject": subj,
        "n_beats": len(vals),
        "mean_rri_ms": np.mean(vals),
        "sdnn_ms": np.std(vals),
        "rmssd_ms": np.sqrt(np.mean(np.diff(vals) ** 2)),
        "pnn50_pct": pnn50(vals),
        "mean_hr_bpm": 60000.0 / np.mean(vals),
        "recording_duration_min": (g["time_s"].max() - g["time_s"].min()) / 60.0,
    })
rri_profile = pd.DataFrame(rows).sort_values(
    "subject", key=lambda s: s.str.extract(r"(\d+)")[0].astype(int)
).reset_index(drop=True)
rri_profile.to_csv(PROCESSED_DIR / "rri_subject_profile.csv", index=False)
print(rri_profile.shape)
rri_profile

(23, 8)


,subject,n_beats,mean_rri_ms,sdnn_ms,rmssd_ms,pnn50_pct,mean_hr_bpm,recording_duration_min
0,p1,35974,952.795885,132.584486,12.203245,0.144553,62.972564,149.887500
1,p2,33789,804.744184,69.994513,12.877839,0.112466,74.557855,140.783333
2,p3,30890,721.704345,211.228932,13.613857,0.848198,83.136537,128.704166
3,p4,26655,1040.232213,135.913557,21.228042,2.138516,57.679429,111.058333
4,p5,31006,846.949363,141.565738,16.117432,1.093372,70.842488,129.187500
5,p6,33604,961.377641,142.112834,18.648291,1.276672,62.410438,140.012499
6,p7,29805,848.309676,102.999173,13.211250,0.208026,70.728888,124.183334
7,p9,30578,874.550759,115.272432,15.489993,1.066161,68.606652,127.404167
8,p10,30575,904.929356,205.235095,18.349889,1.625564,66.303518,127.391667
9,p12,34395,771.680560,120.644079,11.490202,0.375065,77.752380,143.308333
